In [ ]:
# %load warpdisk.py
################################################################################
###### This is an example script to generate HDF5-format ICs for GIZMO
######  The specific example below is obviously arbitrary, but could be generalized
######  to whatever IC you need. 
################################################################################
################################################################################


## load libraries we will use 
import numpy as np
import h5py as h5py
import math as m
import matplotlib.pyplot as plt



def taper1(x,x0,sharp): #default setting center of taper x0=3*rin
    f = 1.0 - 1.0/(m.exp((x-x0)/sharp)+1.)
    return f
def taper2(x,rin):
    f =1.- np.power(rin/x,3)
    return f
    
    
def dr_taper1(x,x0,sharp):
    dfdr = 1.0/(m.exp((x-x0)/sharp)+1.)/(m.exp((x-x0)/sharp)+1.)/sharp*m.exp((x-x0)/sharp)
    return dfdr
def dr_taper2(x,rin,rout):
    dfdr = -0.5*(m.sqrt(rin)+m.sqrt(rout))*m.pow(x,-1.5)+m.sqrt(rin*rout)/x/x
    return dfdr

def Polar2Cartisian(radius,theta):
    xx=np.zeros(len(radius))
    yy = np.zeros(len(radius))
    for i in range(0,len(radius)):
        xx[i] = radius[i] * m.cos(theta[i])
        yy[i] = radius[i] * m.sin(theta[i])
    return xx, yy

def Veltransform(vr,vt,theta):
    vx = np.zeros(len(theta))
    vy = np.zeros(len(theta))
    for i in range(len(theta)):
        #vx[i] = vr[i] * m.cos(theta[i]) - vt[i] * m.sin(theta[i])
        #vy[i] = vr[i] * m.sin(theta[i]) + vt[i] * m.cos(theta[i])
        vx[i] =  - vt[i] * m.sin(theta[i])
        vy[i] =  vt[i] * m.cos(theta[i])
    return vx, vy

def phi(rr,zz):
    potential=-1./m.sqrt(rr*rr+zz*zz)
    return potential
def randparticle(numpart,rho_power,temp_power,rin,rout,h0,c_s):
    import random
    step = 1.0/10000 * (rout - rin)
    sigma = np.zeros(10000)
    rho = np.zeros(10000)
    rr = np.arange(rin,rout,step)
    maxrho = 0
    count = 0
    radius = []
    z = []
    normal = 0
    normal1 = 0
    Hmax = h0 * m.pow((rout/rin),(temp_power+3.0)/2.0) * 8
    aspect0=h0/rin


    for i in range(0,10000):
        sigma[i] = taper2(rr[i],rin)* m.pow(rr[i],rho_power) * 2.0 *np.pi * rr[i] * m.pow((rr[i]/rin),(temp_power+3.0)/2.0) * h0 * m.sqrt(2.0 * np.pi) #note the final term corespond to the geometry factor 2*pi*r*rho
        rho[i] = taper2(rr[i],rin)* m.pow(rr[i]/rin,rho_power) * rr[i]
        normal += step * sigma[i]
    for i in range(0,len(rho)):
        if rho[i] > maxrho:
            maxrho = rho[i]
    maxrho = maxrho *1.1

    while count < numpart :
        x1 = np.random.uniform(rin,rout)
        h1 = h0 * m.pow((x1/rin),(temp_power+3.0)/2.0)         
#        Hmax= 6* h1
        y1 = np.random.uniform(-Hmax, Hmax)
        z1 = np.random.uniform(0,maxrho)
        #phi0=phi(x1,0)
#        phi1=phi(x1,y1)
        
        #rho_x1_y1 =  x1* m.pow((x1/rin),rho_power)* m.exp((phi0-phi1)/c_s/c_s*m.pow((rr[i]/rin),-temp_power))

        rho_x1_y1 = taper2(x1,rin)*x1*m.pow(x1/rin,rho_power)* m.exp(-y1*y1/2./h1/h1)
#        rho_x1_y1 = x1* m.pow((x1/rin),rho_power)*m.exp(1./aspect0/aspect0 * m.pow((x1/rin),(-temp_power-1.))*(1.0/m.sqrt(1+y1*y1/x1/x1)-1))
 
        if rho_x1_y1 > z1 :
            radius.append(x1)
            z.append(y1)
            count += 1
            if (count % 10000 ==0):
                print (count)

            
    radius=np.asarray(radius)
    z=np.asarray(z)
    print ('maxr=',max(radius),'minr=',min(radius))
        
    theta=np.random.uniform(0,2 * np.pi,numpart)
    return radius,theta,z,normal,normal1


    
    
    
def makedisk():
    fname = 'disk-ics.hdf5'
    Ngas = 2000000
#    mu=1.0
    gamma=1.001
    selfgravity = 0 # selfgravity = 0 no dark particle in ic
    bndparticle = 0
    StarMass = 1.
    Rho_Power =-2 #
    Temp_Power = -1
    Sigma_Power = Rho_Power+(Temp_Power+3.0)/2.0
    Rin = 1
    Rout = 50. #in code unit corespond to 62.5AU
    diskmass = 0.05
#    soft =0.02
    Sigmain= diskmass/2/np.pi/(Rout-Rin)/Rin # will change as sigmapower changes
    V_k = m.sqrt(StarMass/Rin)
    Qout=1.2
    Qin=Qout*np.power((Rin/Rout),(-Sigma_Power-1.5+Temp_Power/2.))
    C_s = 0.02*np.sqrt(1/Rin)
#    T0=280
 #   C_s=0.003*np.sqrt(280/2.4)
    H0 = C_s / V_k
    Aspect0 =H0/Rin
    aspect1=H0*m.pow((5/Rin),(Temp_Power+3.0)/2.0) / 5 

    print ('aspect radio at 5 au', aspect1,"at rin", Aspect0)
    print ('selfgravity=',selfgravity,'diskmass/msun=',diskmass)
    print ('sigma_power=', Sigma_Power,'sigmain=',Sigmain)

    print ('Qin=',Qin,'Cs=',C_s)
    
    Radius,Theta,Z, Normal, Normal1 = randparticle(Ngas,Rho_Power,Temp_Power,Rin,Rout,H0,C_s)
    print ('normal=',Normal, 'normal1=',Normal1)
    print ('rho=', diskmass/Normal)
    print ('from',Rin , 'to',Rout,'zmax=',max(Z))

    #  first - open the hdf5 ics file, with the desired filename
    file = h5py.File(fname,'w') 
    # set particle number of each type into the 'npart' vector
    #  NOTE: this MUST MATCH the actual particle numbers assigned to each type, i.e.
    #   npart = np.array([number_of_PartType0_particles,number_of_PartType1_particles,number_of_PartType2_particles,
    #                     number_of_PartType3_particles,number_of_PartType4_particles,number_of_PartType5_particles])
    #   or else the code simply cannot read the IC file correctly!
    #
    if selfgravity == 0 :
        npart = np.array([Ngas,0,0,0,0,0]) # we have gas and particles we will set for type 3 here, zero for all others
    else:
        npart = np.array([Ngas,1,0,0,0,0])
        
    # now we make the Header - the formatting here is peculiar, for historical (GADGET-compatibility) reasons
    h = file.create_group("Header");
    # here we set all the basic numbers that go into the header
    # (most of these will be written over anyways if it's an IC file; the only thing we actually *need* to be 'correct' is "npart")
    h.attrs['NumPart_ThisFile'] = npart; # npart set as above - this in general should be the same as NumPart_Total, it only differs 
                                         #  if we make a multi-part IC file. with this simple script, we aren't equipped to do that.
    h.attrs['NumPart_Total'] = npart; # npart set as above
    h.attrs['NumPart_Total_HighWord'] = 0*npart; # this will be set automatically in-code (for GIZMO, at least)
    h.attrs['MassTable'] = np.zeros(6); # these can be set if all particles will have constant masses for the entire run. however since 
                                        # we set masses explicitly by-particle this should be zero. that is more flexible anyways, as it 
                                        # allows for physics which can change particle masses 
    ## all of the parameters below will be overwritten by whatever is set in the run-time parameterfile if
    ##   this file is read in as an IC file, so their values are irrelevant. they are only important if you treat this as a snapshot
    ##   for restarting. Which you shouldn't - it requires many more fields be set. But we still need to set some values for the code to read
    h.attrs['Time'] = 0.0;  # initial time
    h.attrs['Redshift'] = 0.0; # initial redshift
    h.attrs['BoxSize'] = 10.0; # box size
    h.attrs['NumFilesPerSnapshot'] = 1; # number of files for multi-part snapshots
    h.attrs['Omega0'] = 0.0; # z=0 Omega_matter
    h.attrs['OmegaLambda'] = 0.0; # z=0 Omega_Lambda
    h.attrs['HubbleParam'] = 0.0; # z=0 hubble parameter (small 'h'=H/100 km/s/Mpc)
    h.attrs['Flag_Sfr'] = 0; # flag indicating whether star formation is on or off
    h.attrs['Flag_Cooling'] = 0; # flag indicating whether cooling is on or off
    h.attrs['Flag_StellarAge'] = 0; # flag indicating whether stellar ages are to be saved
    h.attrs['Flag_Metals'] = 0; # flag indicating whether metallicity are to be saved
    h.attrs['Flag_Feedback'] = 0; # flag indicating whether some parts of springel-hernquist model are active
    h.attrs['Flag_DoublePrecision'] = 0; # flag indicating whether ICs are in single/double precision
    h.attrs['Flag_IC_Info'] = 0; # flag indicating extra options for ICs
    ## ok, that ends the block of 'useless' parameters
    
    # Now, the actual data!
    #   These blocks should all be written in the order of their particle type (0,1,2,3,4,5)
    #   If there are no particles of a given type, nothing is needed (no block at all)
    #   PartType0 is 'special' as gas. All other PartTypes take the same, more limited set of information in their ICs
    
    # start with particle type zero. first (assuming we have any gas particles) create the group 
    p = file.create_group("PartType0")
    # now combine the xyz positions into a matrix with the correct format
    q=np.zeros((Ngas,3))

    q[:,2] = Z
    xx,yy = Polar2Cartisian(Radius, Theta)
    q[:,0] = xx
    q[:,1] = yy
    xpos=diskmass/Ngas*xx.sum()
    ypos=diskmass/Ngas*yy.sum()
    print ("xpos=",xpos,"ypos=",ypos)
    # write it to the 'Coordinates' block
    p.create_dataset("Coordinates",data=q)
    # similarly, combine the xyz velocities into a matrix with the correct format
    q=np.zeros((Ngas,3))
    q[:,2] = 0
    Vr = np.zeros(Ngas)
    Vt = np.zeros(Ngas)
    id_g=np.arange(1,Ngas+1)
    for i in range(0,len(Radius)):
        Vt[i] = V_k * V_k * Rin / Radius[i]  
#                + C_s * C_s * Temp_Power * m.pow((Radius[i]/Rin),Temp_Power) \
 #               +  C_s * C_s * m.pow((Radius[i]/Rin),Temp_Power) * Rho_Power \
  #              + (phi(Radius[i],Z[i])-phi(Radius[i],0))*Temp_Power
          
      #          + 2*diskmass*(m.pow(Radius[i],Sigma_Power+2)-m.pow(Rin,Sigma_Power+2))/(m.pow(Rout,Sigma_Power+2)-m.pow(Rin,Sigma_Power+2))\
                
        
        Vt[i] = np.sqrt(Vt[i])
                
    Vx,Vy = Veltransform(Vr,Vt,Theta)

    q[:,0] = Vx
    q[:,1] = Vy
    px=diskmass/Ngas* Vx.sum()
    py=diskmass/Ngas* Vy.sum()

    print ("px=",px,"py=",py)
    # write it to the 'Velocities' block
    p.create_dataset("Velocities",data=q)
    # write particle ids to the ParticleIDs block
    
    array = np.asarray(Radius)
    order = array.argsort()
    ranks = order.argsort()

    p.create_dataset("ParticleIDs",data=ranks)
    # write particle masses to the Masses block
    mass=np.zeros(Ngas)

    mass[:] = diskmass / Ngas 
    p.create_dataset("Masses",data=mass)
    # write internal energies to the InternalEnergy block
    internalu=np.zeros(Ngas)
    for i in range(0,Ngas):
        internalu[i] = C_s * C_s /gamma/(gamma-1) * m.pow((Radius[i]/Rin), Temp_Power)

    p.create_dataset("InternalEnergy",data=internalu)
    print ('maxu=',max(internalu))
    # combine the xyz magnetic fields into a matrix with the correct format
    #   obvious reasons). 

    if selfgravity > 0:
        p = file.create_group("PartType1")
        id_d=np.arange(Ngas+1,Ngas+2)
        id_d[:]=50000000
        p.create_dataset("ParticleIDs",data=id_d)
        mv_d=np.zeros(1)
        mv_d[0] =StarMass
    #    mass = np.asarray(mv_d)
        p.create_dataset("Masses",data=mv_d)
    
        q=np.zeros((1,3))
        q[0][0]=-xpos
        q[0][1]=-ypos
        p.create_dataset("Coordinates",data=q)
        q[0][0]=-px
        q[0][1]=-py
        p.create_dataset("Velocities",data=q)


    file.close()
    # no PartType4 for this IC
    # no PartType5 for this IC

    # close the HDF5 file, which saves these outputs



makedisk()


aspect radio at 5 au 0.02 at rin 0.02
selfgravity= 0 diskmass/msun= 0.05
sigma_power= -1.0 sigmain= 0.00016240300315499524
Qin= 60.0 Cs= 0.02
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
